# 📊 Notebook 01: Pengenalan Data & Tipe Data
## Analitika Data Keuangan Sektor Publik

**Tujuan Pembelajaran:**
1. Memuat dan mengeksplorasi dataset keuangan daerah
2. Mengidentifikasi tipe data setiap kolom
3. Memahami statistik deskriptif
4. Membedakan fitur (feature) dan target (label)
5. Memvisualisasikan distribusi data

---
> **Dataset:** `keuangan_pemda.csv` — Data keuangan pemerintah daerah (Anggaran & Realisasi APBD)

## 📦 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('Library berhasil dimuat ✅')

## 📂 2. Memuat Dataset

In [ ]:
# Muat dataset keuangan pemda
# Sesuaikan path sesuai lokasi file
df = pd.read_csv('../../Dataset/01-Pengenalan Data/iris.csv')

print('Dataset Iris:')
print(f'  Dimensi : {df.shape[0]} baris × {df.shape[1]} kolom')
df.head()

In [ ]:
# Muat dataset keuangan pemda
df_apbd = pd.read_csv('../../Dataset/02-EDA/keuangan_pemda.csv')

print('Dataset Keuangan PEMDA:')
print(f'  Dimensi : {df_apbd.shape[0]} baris × {df_apbd.shape[1]} kolom')
df_apbd.head(10)

## 🔍 3. Eksplorasi Awal Dataset

In [ ]:
# Informasi umum dataset
print('=' * 60)
print('INFORMASI DATASET KEUANGAN PEMDA')
print('=' * 60)
df_apbd.info()

In [ ]:
# Statistik deskriptif untuk kolom numerik
print('STATISTIK DESKRIPTIF')
print('-' * 60)
df_apbd.describe().applymap(lambda x: f'{x:,.0f}' if isinstance(x, float) else x)

In [ ]:
# Cek missing values
print('MISSING VALUES PER KOLOM')
print('-' * 40)
missing = df_apbd.isnull().sum()
missing_pct = (df_apbd.isnull().sum() / len(df_apbd)) * 100

missing_df = pd.DataFrame({
    'Jumlah Missing': missing,
    'Persentase (%)': missing_pct.round(2)
})
print(missing_df[missing_df['Jumlah Missing'] > 0])

## 🗂️ 4. Identifikasi Tipe Data

Setiap kolom memiliki tipe data yang berbeda. Mari kita identifikasi:

In [ ]:
# Buat tabel ringkasan tipe data
tipe_data_info = {
    'Kolom': df_apbd.columns.tolist(),
    'Dtype Python': df_apbd.dtypes.tolist(),
    'Tipe Statistik': [
        'Nominal',      # Kode_Pemda
        'Diskrit',      # Tahun
        'Rasio',        # Anggaran_APBD
        'Rasio',        # Realisasi_APBD
        'Rasio',        # PAD
        'Rasio',        # Dana_Transfer
        'Rasio',        # Belanja_Pegawai
        'Rasio',        # Belanja_Barang
        'Rasio',        # Belanja_Modal
        'Rasio',        # Belanja_Lainnya
        'Rasio',        # Total_Pendapatan
        'Ordinal'       # Predikat
    ],
    'Peran dalam Model': [
        'Identifier',   # Kode_Pemda
        'Feature',      # Tahun
        'Feature',      # Anggaran_APBD
        'Feature',      # Realisasi_APBD
        'Feature',      # PAD
        'Feature',      # Dana_Transfer
        'Feature',      # Belanja_Pegawai
        'Feature',      # Belanja_Barang
        'Feature',      # Belanja_Modal
        'Feature',      # Belanja_Lainnya
        'Feature',      # Total_Pendapatan
        'TARGET/LABEL'  # Predikat
    ]
}

tipe_df = pd.DataFrame(tipe_data_info)
print(tipe_df.to_string(index=False))

## 📊 5. Visualisasi Distribusi Data

In [ ]:
# Distribusi Predikat Kinerja (Variabel Target - Ordinal)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

predikat_order = ['Kurang', 'Cukup', 'Baik', 'Sangat Baik']
colors = ['#e74c3c', '#f39c12', '#2ecc71', '#27ae60']

# Bar chart
counts = df_apbd['Predikat'].value_counts()
counts = counts.reindex([p for p in predikat_order if p in counts.index])
bars = axes[0].bar(counts.index, counts.values, color=colors[:len(counts)], edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribusi Predikat Kinerja PEMDA', fontweight='bold', pad=15)
axes[0].set_xlabel('Predikat')
axes[0].set_ylabel('Jumlah PEMDA')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(val), ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, colors=colors[:len(counts)],
            autopct='%1.1f%%', startangle=90, pctdistance=0.85)
axes[1].set_title('Proporsi Predikat Kinerja', fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('distribusi_predikat.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nCatatan: Predikat adalah variabel ORDINAL — ada urutan (Kurang < Cukup < Baik < Sangat Baik)')

In [ ]:
# Distribusi Anggaran APBD (Variabel Numerik - Rasio)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Filter nilai tidak wajar untuk visualisasi awal
df_valid = df_apbd[df_apbd['Anggaran_APBD'] > 0].dropna(subset=['Anggaran_APBD'])
anggaran_miliar = df_valid['Anggaran_APBD'] / 1e9

axes[0].hist(anggaran_miliar, bins=20, color='#3498db', edgecolor='white', linewidth=1.2)
axes[0].set_title('Distribusi Anggaran APBD\n(nilai valid, dalam Miliar Rp)', fontweight='bold')
axes[0].set_xlabel('Anggaran (Miliar Rp)')
axes[0].set_ylabel('Frekuensi')
axes[0].axvline(anggaran_miliar.mean(), color='red', linestyle='--', label=f'Mean: {anggaran_miliar.mean():.1f} M')
axes[0].axvline(anggaran_miliar.median(), color='orange', linestyle='--', label=f'Median: {anggaran_miliar.median():.1f} M')
axes[0].legend()

# Boxplot
axes[1].boxplot(anggaran_miliar, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#3498db', alpha=0.7))
axes[1].set_title('Boxplot Anggaran APBD\n(deteksi outlier)', fontweight='bold')
axes[1].set_ylabel('Anggaran (Miliar Rp)')
axes[1].set_xticks([1])
axes[1].set_xticklabels(['Anggaran_APBD'])

plt.tight_layout()
plt.show()
print(f'Mean   : Rp {anggaran_miliar.mean():.2f} Miliar')
print(f'Median : Rp {anggaran_miliar.median():.2f} Miliar')
print(f'Std    : Rp {anggaran_miliar.std():.2f} Miliar')

In [ ]:
# Scatter plot: Anggaran vs Realisasi (hubungan antar feature)
fig, ax = plt.subplots(figsize=(10, 6))

df_clean = df_apbd[(df_apbd['Anggaran_APBD'] > 0) & 
                   (df_apbd['Realisasi_APBD'] > 0)].dropna(subset=['Anggaran_APBD', 'Realisasi_APBD', 'Predikat'])

color_map = {'Kurang': '#e74c3c', 'Cukup': '#f39c12', 'Baik': '#2ecc71', 'Sangat Baik': '#27ae60'}

for predikat, color in color_map.items():
    mask = df_clean['Predikat'] == predikat
    if mask.sum() > 0:
        ax.scatter(df_clean.loc[mask, 'Anggaran_APBD'] / 1e9,
                   df_clean.loc[mask, 'Realisasi_APBD'] / 1e9,
                   c=color, label=predikat, alpha=0.7, s=60, edgecolors='white')

# Garis diagonal (realisasi = anggaran = 100%)
max_val = max(df_clean['Anggaran_APBD'].max(), df_clean['Realisasi_APBD'].max()) / 1e9
ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='Realisasi = 100%')

ax.set_xlabel('Anggaran APBD (Miliar Rp)', fontsize=12)
ax.set_ylabel('Realisasi APBD (Miliar Rp)', fontsize=12)
ax.set_title('Scatter Plot: Anggaran vs Realisasi APBD\n(warna = Predikat / Label)', fontweight='bold', fontsize=13)
ax.legend(title='Predikat Kinerja', loc='upper left')
plt.tight_layout()
plt.show()
print('\nObservasi:')
print('- Titik di atas garis diagonal: Realisasi > Anggaran (over-realisasi)')
print('- Titik merah (Kurang) cenderung jauh di bawah garis diagonal')
print('- Titik hijau (Sangat Baik) mendekati garis diagonal')

## 🎯 6. Konsep Feature dan Target

In [ ]:
# Pisahkan fitur (X) dan target (y)
# Kolom identifier: Kode_Pemda
# Target (Label): Predikat
# Features: semua kolom numerik

feature_cols = ['Anggaran_APBD', 'Realisasi_APBD', 'PAD',
                'Dana_Transfer', 'Belanja_Pegawai', 'Belanja_Barang',
                'Belanja_Modal', 'Belanja_Lainnya', 'Total_Pendapatan']

target_col = 'Predikat'

X = df_apbd[feature_cols]
y = df_apbd[target_col]

print('=' * 50)
print('PEMISAHAN FEATURE DAN TARGET')
print('=' * 50)
print(f'Jumlah Feature (X)  : {X.shape[1]} kolom')
print(f'Jumlah Instance (X) : {X.shape[0]} baris')
print(f'Target (y)          : kolom "{target_col}"')
print(f'\nDistribusi Target:')
print(y.value_counts())

In [ ]:
# Heatmap korelasi antar feature
df_numeric = df_apbd[feature_cols].dropna()
correlation = df_numeric.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, center=0, square=True, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Korelasi'})
ax.set_title('Heatmap Korelasi Antar Feature\n(Data Keuangan PEMDA)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()
print('\nInterpretasi:')
print('- Korelasi mendekati +1 (hijau): Hubungan positif kuat')
print('- Korelasi mendekati -1 (merah): Hubungan negatif kuat')
print('- Korelasi mendekati 0: Tidak ada hubungan linear')

## 📝 7. Latihan

Jawab pertanyaan berikut berdasarkan eksplorasi data di atas:

1. **Tipe data** apa yang dimiliki kolom `Predikat`? Mengapa?
2. Berapa jumlah **missing values** pada dataset? Kolom mana yang paling banyak kosong?
3. Apakah ada **nilai tidak wajar** (misalnya negatif) dalam dataset? Di kolom mana?
4. Dari heatmap korelasi, **feature mana** yang memiliki korelasi paling tinggi dengan `Realisasi_APBD`?
5. Jika ingin **memprediksi Predikat Kinerja**, apa jenis tugas Data Mining yang sesuai?

---
*(Tulis jawaban di sini atau di lembar terpisah)*

In [ ]:
# ✏️ Ruang Eksplorasi Mandiri
# Lakukan eksplorasi tambahan di sini!

# Contoh: Cek persentase realisasi
df_apbd_clean = df_apbd[(df_apbd['Anggaran_APBD'] > 0) & (df_apbd['Realisasi_APBD'].notna())].copy()
df_apbd_clean['Pct_Realisasi'] = (df_apbd_clean['Realisasi_APBD'] / df_apbd_clean['Anggaran_APBD'] * 100).round(2)

print('Statistik Persentase Realisasi APBD:')
print(df_apbd_clean['Pct_Realisasi'].describe())